# CholBindNet pocket complexity analysis with dpocket

This notebook:

1. Scans a directory of holo PDB files containing experimentally validated cholesterol.
2. Builds a `dpocket` input file automatically.
3. Runs `dpocket` on those PDBs.
4. Parses the `explicitp` output to extract pocket descriptors such as **volume**.
5. Computes a custom **residue density** metric:

\[
\text{residue density} = \frac{\text{number of unique protein residues within cutoff of cholesterol}}{\text{pocket volume}}
\]

6. Saves summary tables and creates figures that can be used in the CholBindNet revision.

The notebook is written to be flexible because `dpocket` output column names can vary slightly by version.

## What to use for the revision

For the main revision, run this on the **experimentally validated cholesterol-bound pockets**.

That is the cleanest dataset for supporting claims about the complexity of real cholesterol binding sites.

You can optionally run the same workflow on Vina or decoy pockets later as a supplementary comparison, but the main analysis should be based on experimentally validated sites.

In [1]:

# --- Imports ---
import os
import re
import csv
import math
import shlex
import json
import glob
import shutil
import subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
# --- User configuration ---

# Directory containing holo PDB files with experimentally validated cholesterol bound
PDB_DIR = Path("/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB")

# Directory containing random-pocket / decoy / non-cholesterol PDB files
RANDOM_PDB_DIR = Path("/home/alexhernandez/CholBindNet/3DCNN/RANDOM-PDB")  # <-- change this
RANDOM_SOURCE_PROTEIN_DIR = Path("/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB")

# Output directory for all dpocket outputs, plots, and CSV files
OUT_DIR = Path("./dpocket_cholbindnet_analysis")

# Name/path of the dpocket executable
DPOCKET_EXE = "/usr/local/bin/dpocket"
FPOCKET_EXE = "/usr/local/bin/fpocket"

# Ligand residue names to treat as cholesterol
CHOLESTEROL_RESNAMES = ["CLR", "CHL1", "CHL", "STE", "CHO"]

# Distance cutoff in Angstroms for defining contacting protein residues
CONTACT_CUTOFF = 5.0

# Maximum number of random pockets to analyze
MAX_RANDOM_POCKETS = 700
MAX_RANDOM_SOURCE_PROTEINS = 700
GENERATE_RANDOM_POCKET_DIR = False
FPOCKET_RANDOM_OUT_DIR = OUT_DIR / "fpocket_random_outputs"

# Random seed for reproducibility
RANDOM_SEED = 42

# Figure DPI
FIG_DPI = 300

# Reuse already-saved cholesterol results instead of rerunning
LOAD_EXISTING_CHOLESTEROL_RESULTS = False
CHOLESTEROL_RESULTS_CSV = OUT_DIR / "cholesterol_pocket_complexity_metrics.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDB_DIR:", PDB_DIR)
print("RANDOM_PDB_DIR:", RANDOM_PDB_DIR)
print("OUT_DIR:", OUT_DIR.resolve())

PDB_DIR: /home/alexhernandez/CholBindNet/3DCNN/CLR-PDB
RANDOM_PDB_DIR: /home/alexhernandez/CholBindNet/3DCNN/RANDOM-PDB
OUT_DIR: /home/alexhernandez/CholBindNet/FPocketGNN/dpocket_cholbindnet_analysis


In [3]:
def check_optional_executable(exe_name: str):
    resolved = shutil.which(exe_name) if os.path.sep not in exe_name else exe_name
    if resolved and Path(resolved).exists():
        print(f"Found executable: {resolved}")
        return resolved
    raise FileNotFoundError(f"Could not find executable '{exe_name}'.")

fpocket_path = check_optional_executable(FPOCKET_EXE)

# --- Helper: verify dpocket exists ---

def parse_pdb_atoms(pdb_path: Path):
    '''
    Parse ATOM/HETATM records from a PDB file into a DataFrame.

    Returned columns:
      record_name, atom_name, altloc, resname, chain, resseq, icode,
      x, y, z, element
    '''
    rows = []
    with open(pdb_path, "r", encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                continue

            record_name = line[0:6].strip()
            atom_name    = line[12:16].strip()
            altloc       = line[16:17].strip()
            resname      = line[17:20].strip()
            chain        = line[21:22].strip()
            resseq       = line[22:26].strip()
            icode        = line[26:27].strip()
            x            = line[30:38].strip()
            y            = line[38:46].strip()
            z            = line[46:54].strip()
            element      = line[76:78].strip()

            # Keep blank or A altlocs only
            if altloc not in ("", "A"):
                continue

            try:
                x = float(x)
                y = float(y)
                z = float(z)
            except ValueError:
                continue

            rows.append({
                "record_name": record_name,
                "atom_name": atom_name,
                "resname": resname,
                "chain": chain,
                "resseq": resseq,
                "icode": icode,
                "x": x,
                "y": y,
                "z": z,
                "element": element
            })

    return pd.DataFrame(rows)

def residue_uid(df_row):
    return (df_row["chain"], str(df_row["resseq"]).strip(), df_row["icode"], df_row["resname"])

def check_executable(exe_name: str):
    resolved = shutil.which(exe_name) if os.path.sep not in exe_name else exe_name
    if resolved and Path(resolved).exists():
        print(f"Found dpocket: {resolved}")
        return resolved
    raise FileNotFoundError(
        f"Could not find dpocket executable '{exe_name}'. "
        "Edit DPOCKET_EXE at the top of the notebook."
    )

dpocket_path = check_executable(DPOCKET_EXE)

def load_dpocket_table(path: Path):
    if not path.exists():
        raise FileNotFoundError(path)

    try:
        df = pd.read_csv(path, sep="\t", comment="#")
        if df.shape[1] > 1:
            df.columns = [str(c).strip() for c in df.columns]
            return df
    except Exception:
        pass

    try:
        df = pd.read_csv(path, sep=r"\s+", engine="python", comment="#")
        if df.shape[1] > 1:
            df.columns = [str(c).strip() for c in df.columns]
            return df
    except Exception:
        pass

    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = [ln.strip() for ln in fh if ln.strip() and not ln.startswith("#")]

    header = re.split(r"\s+", lines[0])
    data = [re.split(r"\s+", ln) for ln in lines[1:]]
    df = pd.DataFrame(data, columns=header[:len(data[0])])
    df.columns = [str(c).strip() for c in df.columns]
    return df


def find_column(df, candidate_names=None, regex_patterns=None):
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}

    if candidate_names:
        for cand in candidate_names:
            if cand.lower() in lower_map:
                return lower_map[cand.lower()]

    if regex_patterns:
        for pat in regex_patterns:
            for c in cols:
                if re.search(pat, c, flags=re.I):
                    return c

    return None


def stem_from_any_path(x):
    if pd.isna(x):
        return pd.NA
    return Path(str(x).strip()).stem


def pairwise_min_distances(A, B):
    if len(A) == 0 or len(B) == 0:
        return np.array([])
    diff = A[:, None, :] - B[None, :, :]
    d2 = np.sum(diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1))


def compute_contact_metrics_for_ligand_pdb(
    pdb_path: Path,
    ligand_resnames,
    cutoff=5.0,
):
    """
    For real cholesterol-bound structures:
    count unique protein residues with at least one atom within cutoff
    of any ligand atom.
    """
    atoms = parse_pdb_atoms(pdb_path)
    if atoms.empty:
        return None

    protein = atoms[atoms["record_name"] == "ATOM"].copy()

    if isinstance(ligand_resnames, str):
        ligand = atoms[
            (atoms["record_name"] == "HETATM") &
            (atoms["resname"] == ligand_resnames)
        ].copy()
    else:
        ligand = atoms[
            (atoms["record_name"] == "HETATM") &
            (atoms["resname"].isin(ligand_resnames))
        ].copy()

    if protein.empty or ligand.empty:
        return None

    lig_xyz = ligand[["x", "y", "z"]].to_numpy(dtype=float)
    prot_xyz = protein[["x", "y", "z"]].to_numpy(dtype=float)

    min_d = pairwise_min_distances(prot_xyz, lig_xyz)
    protein = protein.copy()
    protein["min_dist_to_region"] = min_d

    contacting_atoms = protein[protein["min_dist_to_region"] <= cutoff].copy()

    unique_residues = {
        residue_uid(row)
        for _, row in contacting_atoms.iterrows()
    }

    return {
        "pdb_file": str(pdb_path),
        "pdb_stem": pdb_path.stem,
        "n_protein_atoms_contacting": int(len(contacting_atoms)),
        "n_unique_pocket_residues": int(len(unique_residues)),
        "contact_cutoff_A": float(cutoff)
    }


def compute_contact_metrics_for_random_pocket(
    source_protein_path: Path,
    pocket_pdb_path: Path,
    cutoff=5.0,
):
    """
    For random-pocket analysis:
    count unique protein residues in the full source protein that lie within
    cutoff of the selected random pocket atoms.
    """
    protein_atoms = parse_pdb_atoms(source_protein_path)
    pocket_atoms = parse_pdb_atoms(pocket_pdb_path)

    if protein_atoms.empty or pocket_atoms.empty:
        return None

    protein = protein_atoms[protein_atoms["record_name"] == "ATOM"].copy()
    if protein.empty:
        return None

    pocket_xyz = pocket_atoms[["x", "y", "z"]].to_numpy(dtype=float)
    if len(pocket_xyz) == 0:
        return None

    prot_xyz = protein[["x", "y", "z"]].to_numpy(dtype=float)
    min_d = pairwise_min_distances(prot_xyz, pocket_xyz)

    protein = protein.copy()
    protein["min_dist_to_region"] = min_d

    contacting_atoms = protein[protein["min_dist_to_region"] <= cutoff].copy()

    unique_residues = {
        residue_uid(row)
        for _, row in contacting_atoms.iterrows()
    }

    return {
        "pdb_file": str(source_protein_path),
        "pdb_stem": source_protein_path.stem,
        "pocket_file": str(pocket_pdb_path),
        "pocket_stem": pocket_pdb_path.stem,
        "n_protein_atoms_contacting": int(len(contacting_atoms)),
        "n_unique_pocket_residues": int(len(unique_residues)),
        "contact_cutoff_A": float(cutoff)
    }


def run_dpocket_pipeline(
    pdb_inventory_df,
    ligand_col_name,
    out_prefix,
    ligand_resnames_for_contacts=None,
    cutoff=5.0,
    contact_mode="ligand",   # "ligand" or "pocket"
    pocket_file_col=None,    # required when contact_mode="pocket"
):
    dpocket_input_path = OUT_DIR / f"{out_prefix}_dpocket_input.txt"

    with open(dpocket_input_path, "w", encoding="utf-8") as fh:
        for _, row in pdb_inventory_df.iterrows():
            fh.write(f"{row['pdb_file']}\t{row[ligand_col_name]}\n")

    dpocket_prefix = str(OUT_DIR / out_prefix)
    cmd = [dpocket_path, "-f", str(dpocket_input_path), "-o", dpocket_prefix]

    print("Running:", " ".join(shlex.quote(x) for x in cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)

    print("Return code:", result.returncode)
    if result.returncode != 0:
        print(result.stdout[:3000] if result.stdout else "")
        print(result.stderr[:3000] if result.stderr else "")
        raise RuntimeError(f"dpocket failed for prefix {out_prefix}")

    explicitp_path = OUT_DIR / f"{out_prefix}_exp.txt"
    explicit_df_raw = load_dpocket_table(explicitp_path)

    explicit_df = explicit_df_raw.copy()
    text_cols = {"pdb", "lig"}
    for col in explicit_df.columns:
        if col in text_cols:
            explicit_df[col] = explicit_df[col].astype("string")
        else:
            explicit_df[col] = pd.to_numeric(explicit_df[col], errors="coerce")

    name_col = find_column(
        explicit_df,
        candidate_names=["name", "pdb", "pdb_file", "protein", "filename"],
        regex_patterns=[r"^name$", r"pdb", r"file", r"protein"]
    )

    volume_col = find_column(
        explicit_df,
        candidate_names=["volume", "pock_vol", "pocket_volume"],
        regex_patterns=[r"\bvol", r"volume"]
    )

    as_density_col = find_column(
        explicit_df,
        candidate_names=["as_density", "density"],
        regex_patterns=[r"as_?density", r"alpha.*density", r"\bdensity\b"]
    )

    alpha_spheres_col = find_column(
        explicit_df,
        candidate_names=["nb_AS", "nb_asph", "n_asph", "number_of_alpha_spheres"],
        regex_patterns=[r"nb_?as", r"nb_?asph", r"alpha.*sphere", r"n.*asph"]
    )

    if name_col is None:
        raise ValueError(f"Could not detect name column for {out_prefix}")
    if volume_col is None:
        raise ValueError(f"Could not detect volume column for {out_prefix}")

    contact_rows = []
    for _, row in pdb_inventory_df.iterrows():
        if contact_mode == "ligand":
            out = compute_contact_metrics_for_ligand_pdb(
                Path(row["pdb_file"]),
                ligand_resnames=row[ligand_col_name],
                cutoff=cutoff
            )
        elif contact_mode == "pocket":
            if pocket_file_col is None:
                raise ValueError("pocket_file_col is required when contact_mode='pocket'")
            out = compute_contact_metrics_for_random_pocket(
                source_protein_path=Path(row["pdb_file"]),
                pocket_pdb_path=Path(row[pocket_file_col]),
                cutoff=cutoff
            )
        else:
            raise ValueError("contact_mode must be 'ligand' or 'pocket'")

        if out is not None:
            contact_rows.append(out)

    contact_df = pd.DataFrame(contact_rows)

    analysis_df = explicit_df.copy()
    analysis_df["pdb_stem"] = analysis_df[name_col].astype("string").apply(stem_from_any_path)
    analysis_df["pdb_stem"] = analysis_df["pdb_stem"].astype("string").str.strip()

    if not contact_df.empty:
        contact_df["pdb_stem"] = contact_df["pdb_stem"].astype("string").str.strip()

    merged_df = analysis_df.merge(contact_df, on="pdb_stem", how="left")

    merged_df["pocket_volume"] = pd.to_numeric(merged_df[volume_col], errors="coerce")
    merged_df["residue_density"] = (
        merged_df["n_unique_pocket_residues"] / merged_df["pocket_volume"]
    )

    if alpha_spheres_col is not None:
        merged_df["n_alpha_spheres"] = pd.to_numeric(
            merged_df[alpha_spheres_col], errors="coerce"
        )
        merged_df["alpha_sphere_density_per_volume"] = (
            merged_df["n_alpha_spheres"] / merged_df["pocket_volume"]
        )

    if as_density_col is not None:
        merged_df["dpocket_as_density"] = pd.to_numeric(
            merged_df[as_density_col], errors="coerce"
        )

    return merged_df

Found executable: /usr/local/bin/fpocket
Found dpocket: /usr/local/bin/dpocket


In [4]:
import random
import hashlib

def stable_seed_from_name(name: str, base_seed: int = 42) -> int:
    digest = hashlib.md5(name.encode("utf-8")).hexdigest()
    return base_seed + int(digest[:8], 16)

def get_cholesterol_centroid(pdb_path: Path, chol_resnames):
    atoms = parse_pdb_atoms(pdb_path)
    if atoms.empty:
        return None

    chol = atoms[
        (atoms["record_name"] == "HETATM") &
        (atoms["resname"].isin(chol_resnames))
    ].copy()

    if chol.empty:
        return None

    xyz = chol[["x", "y", "z"]].to_numpy(dtype=float)
    return xyz.mean(axis=0)

def get_pocket_centroid(pocket_pdb_path: Path):
    atoms = parse_pdb_atoms(pocket_pdb_path)
    if atoms.empty:
        return None

    xyz = atoms[["x", "y", "z"]].to_numpy(dtype=float)
    if len(xyz) == 0:
        return None

    return xyz.mean(axis=0)

def run_fpocket_on_directory(
    source_dir: Path,
    fpocket_out_dir: Path,
    random_pdb_dir: Path,
    chol_resnames,
    max_source_proteins: int = 700,
    max_rank_to_consider: int = 10,
    random_seed: int = 42,
):
    """
    For each source protein:
      1. Run fpocket
      2. Look at top-ranked pocket files
      3. Exclude the pocket closest to the true cholesterol centroid
      4. Randomly choose one remaining pocket
      5. Store a metadata row that keeps:
         - original protein path
         - selected pocket file path
         - selected rank
         - excluded rank
    """
    fpocket_out_dir.mkdir(parents=True, exist_ok=True)
    random_pdb_dir.mkdir(parents=True, exist_ok=True)

    protein_pdbs = sorted(source_dir.glob("*.pdb"))
    if len(protein_pdbs) == 0:
        raise FileNotFoundError(f"No .pdb files found in {source_dir}")

    if len(protein_pdbs) > max_source_proteins:
        rng = random.Random(random_seed)
        protein_pdbs = sorted(rng.sample(protein_pdbs, max_source_proteins))

    selected_rows = []

    for pdb_path in protein_pdbs:
        local_pdb = fpocket_out_dir / pdb_path.name
        if not local_pdb.exists():
            shutil.copy2(pdb_path, local_pdb)

        cmd = [fpocket_path, "-f", str(local_pdb)]
        print("Running:", " ".join(shlex.quote(x) for x in cmd))
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            print(f"fpocket failed for {pdb_path.name}")
            if result.stderr:
                print(result.stderr[:1000])
            continue

        out_dir = local_pdb.with_name(local_pdb.stem + "_out")
        pockets_dir = out_dir / "pockets"
        if not pockets_dir.exists():
            continue

        pocket_files = sorted(pockets_dir.glob("pocket*_atm.pdb"))
        if len(pocket_files) == 0:
            pocket_files = sorted(pockets_dir.glob("pocket*.pdb"))
        if len(pocket_files) == 0:
            continue

        ranked_pockets = []
        for pocket_file in pocket_files:
            m = re.search(r"pocket(\d+)", pocket_file.stem, flags=re.I)
            if not m:
                continue
            rank = int(m.group(1))
            if rank <= max_rank_to_consider:
                ranked_pockets.append((rank, pocket_file))

        ranked_pockets = sorted(ranked_pockets, key=lambda x: x[0])
        if len(ranked_pockets) == 0:
            continue

        chol_centroid = get_cholesterol_centroid(pdb_path, chol_resnames)

        excluded_rank = None
        excluded_file = None
        candidate_pockets = ranked_pockets.copy()

        # Exclude the pocket closest to the true cholesterol site
        if chol_centroid is not None and len(ranked_pockets) > 1:
            best_idx = None
            best_dist = None

            for i, (rank, pocket_file) in enumerate(ranked_pockets):
                pocket_centroid = get_pocket_centroid(pocket_file)
                if pocket_centroid is None:
                    continue

                dist = float(np.linalg.norm(pocket_centroid - chol_centroid))
                if best_dist is None or dist < best_dist:
                    best_dist = dist
                    best_idx = i

            if best_idx is not None:
                excluded_rank, excluded_file = ranked_pockets[best_idx]
                candidate_pockets = [
                    rp for j, rp in enumerate(ranked_pockets) if j != best_idx
                ]

        if len(candidate_pockets) == 0:
            continue

        rng = random.Random(stable_seed_from_name(pdb_path.stem, random_seed))
        chosen_rank, chosen_pocket = rng.choice(candidate_pockets)

        copied_pocket_name = f"{pdb_path.stem}__{chosen_pocket.stem}.pdb"
        copied_pocket_path = random_pdb_dir / copied_pocket_name
        shutil.copy2(chosen_pocket, copied_pocket_path)

        selected_rows.append({
            "source_protein": str(pdb_path),
            "source_protein_stem": pdb_path.stem,
            "fpocket_output_dir": str(out_dir),
            "selected_random_pocket_file": str(copied_pocket_path),
            "selected_random_pocket_stem": copied_pocket_path.stem,
            "selected_rank": int(chosen_rank),
            "excluded_rank_putative_chol_pocket": excluded_rank,
            "excluded_pocket_file": str(excluded_file) if excluded_file is not None else None,
            "n_top_candidates_before_exclusion": int(len(ranked_pockets)),
            "n_candidates_after_exclusion": int(len(candidate_pockets)),
        })

    return pd.DataFrame(selected_rows)

# %%
def build_random_pocket_inventory_from_index(
    generated_random_df: pd.DataFrame,
    max_random_pockets: int = 700,
    random_seed: int = 42,
):
    """
    Build inventory directly from the fpocket-generated selection table.

    Important:
    We do NOT try to infer a ligand from pocket*_atm.pdb files.
    dpocket should be run on the original source protein, using the
    known cholesterol residue name from that source protein.
    """
    if generated_random_df is None or generated_random_df.empty:
        return pd.DataFrame()

    inv = generated_random_df.copy()

    # dpocket must be run on the original full protein PDB, not on isolated pocket files
    inv["pdb_file"] = inv["source_protein"]

    # Since RANDOM_SOURCE_PROTEIN_DIR is CLR-PDB, these source proteins contain cholesterol.
    # We can recover the exact ligand name from the source protein itself.
    ligand_names = []
    n_chol_atoms = []

    for pdb_file in inv["pdb_file"]:
        atoms = parse_pdb_atoms(Path(pdb_file))
        if atoms.empty:
            ligand_names.append(np.nan)
            n_chol_atoms.append(np.nan)
            continue

        het = atoms[atoms["record_name"] == "HETATM"].copy()
        chol = het[het["resname"].isin(CHOLESTEROL_RESNAMES)].copy()

        if chol.empty:
            ligand_names.append(np.nan)
            n_chol_atoms.append(np.nan)
        else:
            ligand_names.append(chol["resname"].value_counts().index[0])
            n_chol_atoms.append(int(len(chol)))

    inv["ligand_for_dpocket"] = ligand_names
    inv["n_chol_atoms"] = n_chol_atoms

    inv = inv.dropna(subset=["pdb_file", "ligand_for_dpocket"]).copy()

    if len(inv) > max_random_pockets:
        inv = (
            inv.sample(n=max_random_pockets, random_state=random_seed)
              .sort_values(["source_protein_stem", "selected_rank"])
              .reset_index(drop=True)
        )
    else:
        inv = inv.sort_values(["source_protein_stem", "selected_rank"]).reset_index(drop=True)

    return inv

In [5]:

# --- Identify which PDBs contain cholesterol ---

def find_pdbs_with_cholesterol(pdb_dir: Path, chol_resnames):
    pdb_paths = sorted(list(pdb_dir.glob("*.pdb")))
    found = []

    for pdb_path in pdb_paths:
        atoms = parse_pdb_atoms(pdb_path)
        if atoms.empty:
            continue

        het = atoms[atoms["record_name"] == "HETATM"].copy()
        chol = het[het["resname"].isin(chol_resnames)].copy()

        if len(chol) > 0:
            found.append({
                "pdb_file": pdb_path,
                "n_chol_atoms": len(chol),
                "detected_chol_resnames": sorted(chol["resname"].dropna().unique().tolist())
            })

    return pd.DataFrame(found)

pdb_inventory = find_pdbs_with_cholesterol(PDB_DIR, CHOLESTEROL_RESNAMES)
print("PDB files with cholesterol found:", len(pdb_inventory))
display(pdb_inventory.head())

PDB files with cholesterol found: 770


,pdb_file,n_chol_atoms,detected_chol_resnames
0,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]
1,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]
2,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]
3,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,84,[CLR]
4,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]


In [6]:
# --- Cholesterol metrics: load existing if available ---

if LOAD_EXISTING_CHOLESTEROL_RESULTS and CHOLESTEROL_RESULTS_CSV.exists():
    chol_df = pd.read_csv(CHOLESTEROL_RESULTS_CSV)
    print("Loaded existing cholesterol metrics from:", CHOLESTEROL_RESULTS_CSV)
else:
    pdb_inventory = find_pdbs_with_cholesterol(PDB_DIR, CHOLESTEROL_RESNAMES)
    print("PDB files with cholesterol found:", len(pdb_inventory))
    display(pdb_inventory.head())

    chol_inventory = pdb_inventory.copy()
    chol_inventory["ligand_for_dpocket"] = chol_inventory["detected_chol_resnames"].apply(lambda x: x[0])

    chol_df = run_dpocket_pipeline(
        pdb_inventory_df=chol_inventory,
        ligand_col_name="ligand_for_dpocket",
        out_prefix="chol",
        ligand_resnames_for_contacts=CHOLESTEROL_RESNAMES,
        cutoff=CONTACT_CUTOFF,
        contact_mode="ligand",
    )

    chol_df.to_csv(CHOLESTEROL_RESULTS_CSV, index=False)
    print("Saved cholesterol metrics to:", CHOLESTEROL_RESULTS_CSV)

chol_df["group"] = "cholesterol"
display(chol_df.head())

PDB files with cholesterol found: 770


,pdb_file,n_chol_atoms,detected_chol_resnames
0,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]
1,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]
2,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]
3,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,84,[CLR]
4,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,28,[CLR]


Running: /usr/local/bin/dpocket -f dpocket_cholbindnet_analysis/chol_dpocket_input.txt -o dpocket_cholbindnet_analysis/chol
Return code: 0
Saved cholesterol metrics to: dpocket_cholbindnet_analysis/cholesterol_pocket_complexity_metrics.csv


,pdb,lig,overlap,PP-crit,PP-dst,crit4,crit5,crit6,crit6_continue,lig_vol,pock_vol,nb_AS,nb_AS_norm,mean_as_ray,mean_as_solv_acc,apol_as_prop,apol_as_prop_norm,mean_loc_hyd_dens,mean_loc_hyd_dens_norm,hydrophobicity_score,volume_score,polarity_score,polarity_score_norm,charge_score,flex,prop_polar_atm,as_density,as_density_norm,as_max_dst,as_max_dst_norm,drug_score,convex_hull_volume,surf_pol_vdw14,surf_pol_vdw22,surf_apol_vdw14,surf_apol_vdw22,n_abpa,ALA,ARG,ASN,ASP,CYS,GLN,GLU,GLY,HIS,ILE,LEU,LYS,MET,PHE,PRO,SER,THR,TRP,TYR,VAL,pdb_stem,pdb_file,n_protein_atoms_contacting,n_unique_pocket_residues,contact_cutoff_A,pocket_volume,residue_density,n_alpha_spheres,alpha_sphere_density_per_volume,dpocket_as_density,group
0,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,CLR,100.0,1,0.0,1.0,1.0,1,2.0,350.08,941.68,49,0.0,4.16,0.57,0.94,0.0,27.65,0.0,63.63,4.81,6,0.0,1,19.24,16.67,8.08,0.0,23.07,0.0,0.0,230.13,3.21,0.00,54.34,19.11,3,1,0,0,0,0,0,0,1,0,2,7,1,3,2,2,0,1,0,4,3,1LRI_protein,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,71,24,5.0,941.68,0.025486,49,0.052035,8.08,cholesterol
1,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,CLR,100.0,1,0.0,1.0,1.0,1,2.0,266.59,1098.44,77,0.0,4.08,0.54,0.73,0.0,29.68,0.0,51.59,4.97,13,0.0,3,19.75,22.99,8.49,0.0,20.26,0.0,0.0,463.89,32.34,10.46,47.18,5.73,8,3,2,0,1,3,1,1,0,1,2,2,2,1,5,0,0,0,1,4,3,1N83_protein,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,75,23,5.0,1098.44,0.020939,77,0.070099,8.49,cholesterol
2,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,CLR,100.0,1,0.0,1.0,1.0,1,2.0,308.78,1115.22,72,0.0,3.99,0.49,0.72,0.0,32.62,0.0,47.16,4.84,12,0.0,2,21.19,25.30,8.32,0.0,20.72,0.0,0.0,398.22,23.59,1.74,66.42,9.56,10,1,1,1,0,0,2,1,0,0,4,5,2,0,4,3,1,1,2,1,2,1ZHY_protein,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,77,24,5.0,1115.22,0.021520,72,0.064561,8.32,cholesterol
3,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,CLR,100.0,1,0.0,1.0,1.0,1,2.0,1097.74,1498.72,56,0.0,3.97,0.64,0.80,0.0,12.89,0.0,54.79,4.39,11,0.0,1,75.35,26.04,12.95,0.0,26.67,0.0,0.0,1544.29,58.46,33.29,61.59,21.02,8,2,1,0,0,2,1,1,1,0,4,7,1,0,1,1,2,2,2,1,4,2RH1_protein,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,96,27,5.0,1498.72,0.018015,56,0.037365,12.95,cholesterol
4,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,CLR,100.0,1,0.0,1.0,1.0,1,2.0,358.50,888.00,37,0.0,4.10,0.65,0.76,0.0,13.57,0.0,53.55,4.90,10,0.0,-1,57.19,28.57,8.83,0.0,21.10,0.0,0.0,275.15,21.52,0.00,122.13,26.76,5,0,1,0,0,0,0,2,1,0,4,2,0,1,1,0,1,2,1,3,1,2ZXE_protein,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,40,13,5.0,888.00,0.014640,37,0.041667,8.83,cholesterol


In [7]:
# %%
# --- Generate random pockets and keep the selection index ---

generated_random_df = pd.DataFrame()

if GENERATE_RANDOM_POCKET_DIR:
    generated_random_df = run_fpocket_on_directory(
        source_dir=RANDOM_SOURCE_PROTEIN_DIR,
        fpocket_out_dir=FPOCKET_RANDOM_OUT_DIR,
        random_pdb_dir=RANDOM_PDB_DIR,
        chol_resnames=CHOLESTEROL_RESNAMES,
        max_source_proteins=MAX_RANDOM_SOURCE_PROTEINS,
        max_rank_to_consider=10,
        random_seed=RANDOM_SEED,
    )
    print("Generated random pocket selections:", len(generated_random_df))
    display(generated_random_df.head())

    generated_random_csv = OUT_DIR / "generated_random_pocket_index.csv"
    generated_random_df.to_csv(generated_random_csv, index=False)
    print("Saved:", generated_random_csv)
else:
    generated_random_csv = OUT_DIR / "generated_random_pocket_index.csv"
    if generated_random_csv.exists():
        generated_random_df = pd.read_csv(generated_random_csv)
        print("Loaded existing random pocket index:", generated_random_csv)
        display(generated_random_df.head())
    else:
        raise FileNotFoundError(
            "GENERATE_RANDOM_POCKET_DIR is False and no generated_random_pocket_index.csv was found."
        )

# %%
# --- Build random inventory from the saved fpocket selection index ---

random_inventory = build_random_pocket_inventory_from_index(
    generated_random_df=generated_random_df,
    max_random_pockets=MAX_RANDOM_POCKETS,
    random_seed=RANDOM_SEED,
)

print("Random pockets selected:", len(random_inventory))
display(
    random_inventory[
        [
            "source_protein_stem",
            "selected_rank",
            "excluded_rank_putative_chol_pocket",
            "ligand_for_dpocket",
            "selected_random_pocket_file",
        ]
    ].head()
)

# %%
# --- Run dpocket on source proteins corresponding to random pocket selections ---

print("Unique ligands used for dpocket on random-source proteins:")
print(sorted(random_inventory["ligand_for_dpocket"].dropna().unique().tolist()))

random_df = run_dpocket_pipeline(
    pdb_inventory_df=random_inventory,
    ligand_col_name="ligand_for_dpocket",
    out_prefix="random",
    cutoff=CONTACT_CUTOFF,
    contact_mode="pocket",
    pocket_file_col="selected_random_pocket_file",
)

random_df = random_df.merge(
    random_inventory[
        [
            "source_protein_stem",
            "selected_random_pocket_file",
            "selected_random_pocket_stem",
            "selected_rank",
            "excluded_rank_putative_chol_pocket",
        ]
    ],
    left_on="pdb_stem",
    right_on="source_protein_stem",
    how="left",
)

random_df["group"] = "random"
display(random_df.head())

random_csv = OUT_DIR / "random_pocket_complexity_metrics.csv"
random_df.to_csv(random_csv, index=False)
print("Saved:", random_csv)

Loaded existing random pocket index: dpocket_cholbindnet_analysis/generated_random_pocket_index.csv


,source_protein,source_protein_stem,fpocket_output_dir,selected_random_pocket_file,selected_random_pocket_stem,selected_rank,excluded_rank_putative_chol_pocket,excluded_pocket_file,n_top_candidates_before_exclusion,n_candidates_after_exclusion
0,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,1LRI_protein,dpocket_cholbindnet_analysis/fpocket_random_ou...,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...,1LRI_protein__pocket4_atm,4,1,dpocket_cholbindnet_analysis/fpocket_random_ou...,6,5
1,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,1N83_protein,dpocket_cholbindnet_analysis/fpocket_random_ou...,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...,1N83_protein__pocket4_atm,4,1,dpocket_cholbindnet_analysis/fpocket_random_ou...,10,9
2,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,2RH1_protein,dpocket_cholbindnet_analysis/fpocket_random_ou...,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...,2RH1_protein__pocket2_atm,2,1,dpocket_cholbindnet_analysis/fpocket_random_ou...,10,9
3,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,2ZXE_protein,dpocket_cholbindnet_analysis/fpocket_random_ou...,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...,2ZXE_protein__pocket4_atm,4,2,dpocket_cholbindnet_analysis/fpocket_random_ou...,10,9
4,/home/alexhernandez/CholBindNet/3DCNN/CLR-PDB/...,3A3Y_protein,dpocket_cholbindnet_analysis/fpocket_random_ou...,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...,3A3Y_protein__pocket6_atm,6,4,dpocket_cholbindnet_analysis/fpocket_random_ou...,10,9


Random pockets selected: 700


,source_protein_stem,selected_rank,excluded_rank_putative_chol_pocket,ligand_for_dpocket,selected_random_pocket_file
0,1LRI_protein,4,1,CLR,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...
1,1N83_protein,4,1,CLR,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...
2,2RH1_protein,2,1,CLR,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...
3,2ZXE_protein,4,2,CLR,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...
4,3A3Y_protein,6,4,CLR,/home/alexhernandez/CholBindNet/3DCNN/RANDOM-P...


Unique ligands used for dpocket on random-source proteins:
['CLR']
Running: /usr/local/bin/dpocket -f dpocket_cholbindnet_analysis/random_dpocket_input.txt -o dpocket_cholbindnet_analysis/random


KeyboardInterrupt: 

In [ ]:
# --- Combine cholesterol and random results ---

combined_df = pd.concat(
    [chol_df.copy(), random_df.copy()],
    ignore_index=True
)

combined_df = combined_df.replace([np.inf, -np.inf], np.nan)

combined_csv = OUT_DIR / "cholesterol_vs_random_pocket_complexity_metrics.csv"
combined_df.to_csv(combined_csv, index=False)
print("Saved:", combined_csv)

display(combined_df.head())
print(combined_df["group"].value_counts())

# --- Summary statistics by group ---

def metric_summary(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "std": s.std(ddof=1) if len(s) > 1 else np.nan,
        "min": s.min(),
        "q25": s.quantile(0.25),
        "q75": s.quantile(0.75),
        "max": s.max(),
    })

group_summary = (
    combined_df
    .groupby("group")[["pocket_volume", "n_unique_pocket_residues", "residue_density"]]
    .apply(lambda g: g.apply(metric_summary))
)

display(group_summary)

group_summary.to_csv(OUT_DIR / "group_summary_statistics.csv")
print("Saved:", OUT_DIR / "group_summary_statistics.csv")

In [ ]:
# --- Figure: cholesterol vs random pocket volume ---

fig, ax = plt.subplots(figsize=(6.5, 4.5))

chol_vals = combined_df.loc[combined_df["group"] == "cholesterol", "pocket_volume"].dropna()
rand_vals = combined_df.loc[combined_df["group"] == "random", "pocket_volume"].dropna()

ax.hist(chol_vals, bins=25, alpha=0.6, label="Cholesterol")
ax.hist(rand_vals, bins=25, alpha=0.6, label="Random")

ax.set_xlabel("Pocket volume")
ax.set_ylabel("Count")
ax.set_title("Pocket volume: cholesterol vs random")
ax.legend()

fig.tight_layout()
fig.savefig(OUT_DIR / "figure_compare_pocket_volume_histogram.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: cholesterol vs random residue density ---

fig, ax = plt.subplots(figsize=(6.5, 4.5))

chol_vals = combined_df.loc[combined_df["group"] == "cholesterol", "residue_density"].dropna()
rand_vals = combined_df.loc[combined_df["group"] == "random", "residue_density"].dropna()

ax.hist(chol_vals, bins=25, alpha=0.6, label="Cholesterol")
ax.hist(rand_vals, bins=25, alpha=0.6, label="Random")

ax.set_xlabel("Residue density (unique residues / pocket volume)")
ax.set_ylabel("Count")
ax.set_title("Residue density: cholesterol vs random")
ax.legend()

fig.tight_layout()
fig.savefig(OUT_DIR / "figure_compare_residue_density_histogram.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: boxplots for direct comparison ---

fig, ax = plt.subplots(figsize=(5.8, 4.5))
data = [
    combined_df.loc[combined_df["group"] == "cholesterol", "pocket_volume"].dropna().values,
    combined_df.loc[combined_df["group"] == "random", "pocket_volume"].dropna().values,
]
ax.boxplot(data, labels=["Cholesterol", "Random"])
ax.set_ylabel("Pocket volume")
ax.set_title("Pocket volume comparison")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_compare_pocket_volume_boxplot.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5.8, 4.5))
data = [
    combined_df.loc[combined_df["group"] == "cholesterol", "residue_density"].dropna().values,
    combined_df.loc[combined_df["group"] == "random", "residue_density"].dropna().values,
]
ax.boxplot(data, labels=["Cholesterol", "Random"])
ax.set_ylabel("Residue density")
ax.set_title("Residue density comparison")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_compare_residue_density_boxplot.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()